# 怎样合并多个气象站并按城市统计？：参考答案

每道题先写预期结果再运行。练习中的 TODO 需要自行完成；能顺序运行不代表已完成练习。

In [1]:
from pathlib import Path
CASE_DIR = Path.cwd()
# 载入本例的可修改函数和数据，所有算法都在同目录 solution.py。
exec(compile((CASE_DIR / "solution.py").read_text(encoding="utf-8"), str(CASE_DIR / "solution.py"), "exec"))

  station       date  temp_max         source
0       A 2024-07-01        25  station_A.csv
1       A 2024-07-02        27  station_A.csv
2       A 2024-07-03        29  station_A.csv
3       B 2024-07-01        27  station_B.csv
4       B 2024-07-02        29  station_B.csv
  station city       date  temp_max
0       A   海城 2024-07-01        25
1       A   海城 2024-07-02        27
2       A   海城 2024-07-03        29
3       B   海城 2024-07-01        27
4       B   海城 2024-07-02        29
5       B   海城 2024-07-03        31
6       C   山城 2024-07-01        21
7       C   山城 2024-07-02        23
8       C   山城 2024-07-03        25
      mean   max  count
city                   
山城    23.0  25.0      3
海城    28.0  30.0      3


## 1. 补全：检查连接键
统计每个站点几天有记录，并验证站点元数据唯一。

In [2]:
counts = merged.groupby('station')['date'].nunique()
assert counts.eq(3).all()
assert stations['station'].is_unique
print(counts)

station
A    3
B    3
C    3
Name: date, dtype: int64


## 2. 修改：只统计前两天
筛选日期早于 2024-07-03 的记录，再按城市和日期算均值。

In [3]:
early = merged.loc[merged['date'] < pd.Timestamp('2024-07-03')]
early_summary = early.groupby(['city', 'date'])['temp_max'].mean()
assert len(early) == 6 and len(early_summary) == 4
print(early_summary)

city  date      
山城    2024-07-01    21.0
      2024-07-02    23.0
海城    2024-07-01    26.0
      2024-07-02    28.0
Name: temp_max, dtype: float64


## 3. 迁移：用 SQLite 复算城市日均
可选 SQL 迁移：把已合并数据写入内存数据库，使用 GROUP BY 复算，不建立服务器。

In [4]:
import sqlite3
with sqlite3.connect(':memory:') as connection:
    merged.to_sql('weather', connection, index=False)
    sql_result = pd.read_sql_query('SELECT city, date, AVG(temp_max) AS temp_max FROM weather GROUP BY city, date ORDER BY city, date', connection)
sql_result['date'] = pd.to_datetime(sql_result['date'])
pd.testing.assert_frame_equal(sql_result, city_day.sort_values(['city','date']).reset_index(drop=True))
print(sql_result)

  city       date  temp_max
0   山城 2024-07-01      21.0
1   山城 2024-07-02      23.0
2   山城 2024-07-03      25.0
3   海城 2024-07-01      26.0
4   海城 2024-07-02      28.0
5   海城 2024-07-03      30.0
